# Phase 5: OCR & Document Intelligence
## Day 23: OpenCVPreprocessing

Date: 2026-04-24

### Learning objectives
- Understand why preprocessing improves OCR.
- Convert images to grayscale.
- Use Otsu and adaptive thresholding.
- Denoise noisy document images.
- Deskew tilted text images.
- Build a simple preprocessing pipeline for OCR.

In [ ]:
import json
import math
import re
import textwrap
from pprint import pprint

import numpy as np
import pandas as pd

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    cv2 = None
    CV2_AVAILABLE = False

try:
    from PIL import Image, ImageDraw, ImageFont, ImageFilter
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

def show(title, content):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    print(textwrap.dedent(str(content)).strip())

print("Setup complete.")
print("OpenCV available:", CV2_AVAILABLE)
print("PIL available:", PIL_AVAILABLE)
print("matplotlib available:", MATPLOTLIB_AVAILABLE)

In [ ]:
sample_document_text = '''
Invoice ID: INV-3021
Customer: Berlin Coffee Bar
Campaign: Spring Coffee Push
Spend: 1200 EUR
Clicks: 3420
Conversions: 184
Total: 1200 EUR
'''

print(sample_document_text)

## 1. Create a synthetic document image

We will create a clean document image, then make noisy and skewed versions.

This keeps the notebook self-contained and runnable without external files.

In [ ]:
def create_document_image(text, width=900, height=420, font_size=26):
    if not PIL_AVAILABLE:
        return None

    image = Image.new("RGB", (width, height), color="white")
    draw = ImageDraw.Draw(image)

    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()

    draw.multiline_text((40, 40), text.strip(), fill="black", font=font, spacing=12)
    return image

clean_pil = create_document_image(sample_document_text)

if clean_pil is not None:
    display(clean_pil)
else:
    print("PIL is not available.")

In [ ]:
def pil_to_rgb_array(image):
    if image is None:
        return None
    return np.array(image.convert("RGB"))

def rgb_array_to_pil(array):
    if array is None or not PIL_AVAILABLE:
        return None
    array = np.clip(array, 0, 255).astype(np.uint8)
    return Image.fromarray(array)

clean_rgb = pil_to_rgb_array(clean_pil)

print("Image shape:", None if clean_rgb is None else clean_rgb.shape)
print("Image dtype:", None if clean_rgb is None else clean_rgb.dtype)

In [ ]:
def display_image(array, title=None):
    if array is None:
        print("No image to display.")
        return

    if PIL_AVAILABLE:
        if len(array.shape) == 2:
            display(Image.fromarray(np.clip(array, 0, 255).astype(np.uint8)))
        else:
            display(rgb_array_to_pil(array))
    elif MATPLOTLIB_AVAILABLE:
        if title:
            plt.title(title)
        if len(array.shape) == 2:
            plt.imshow(array, cmap="gray")
        else:
            plt.imshow(array)
        plt.axis("off")
        plt.show()
    else:
        print("Display tools are not available.")

display_image(clean_rgb, "Clean RGB image")

## 2. Grayscale

OCR usually works better after converting to grayscale.

Grayscale removes color information and keeps brightness information.

In [ ]:
def to_grayscale(image_rgb):
    if image_rgb is None:
        return None

    if CV2_AVAILABLE:
        return cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    # Fallback grayscale conversion.
    return np.dot(image_rgb[..., :3], [0.299, 0.587, 0.114]).astype(np.uint8)

gray = to_grayscale(clean_rgb)

print("Gray shape:", None if gray is None else gray.shape)
print("Gray dtype:", None if gray is None else gray.dtype)

display_image(gray, "Grayscale")

In [ ]:
def image_stats(image):
    if image is None:
        return {}
    return {
        "shape": image.shape,
        "min": int(np.min(image)),
        "max": int(np.max(image)),
        "mean": round(float(np.mean(image)), 2),
        "std": round(float(np.std(image)), 2),
    }

pprint(image_stats(gray))

## 3. Global thresholding

Thresholding turns a grayscale image into black and white.

This can make text easier for OCR when the background is clean.

In [ ]:
def global_threshold(gray_image, threshold_value=180):
    if gray_image is None:
        return None

    if CV2_AVAILABLE:
        _, binary = cv2.threshold(gray_image, threshold_value, 255, cv2.THRESH_BINARY)
        return binary

    return np.where(gray_image > threshold_value, 255, 0).astype(np.uint8)

binary_global = global_threshold(gray, threshold_value=180)
display_image(binary_global, "Global threshold")

In [ ]:
threshold_values = [120, 160, 200]

for value in threshold_values:
    binary = global_threshold(gray, threshold_value=value)
    print("Threshold:", value, "Stats:", image_stats(binary))

## 4. Otsu thresholding

Otsu chooses the threshold automatically.

It works well when the image has clear foreground text and background.

In [ ]:
def otsu_threshold(gray_image):
    if gray_image is None:
        return None, None

    if CV2_AVAILABLE:
        value, binary = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary, value

    # Fallback Otsu implementation.
    hist, _ = np.histogram(gray_image.ravel(), bins=256, range=(0, 256))
    total = gray_image.size
    sum_total = np.dot(np.arange(256), hist)

    sum_background = 0
    weight_background = 0
    max_variance = 0
    best_threshold = 0

    for threshold in range(256):
        weight_background += hist[threshold]
        if weight_background == 0:
            continue

        weight_foreground = total - weight_background
        if weight_foreground == 0:
            break

        sum_background += threshold * hist[threshold]
        mean_background = sum_background / weight_background
        mean_foreground = (sum_total - sum_background) / weight_foreground

        variance_between = weight_background * weight_foreground * (mean_background - mean_foreground) ** 2

        if variance_between > max_variance:
            max_variance = variance_between
            best_threshold = threshold

    binary = np.where(gray_image > best_threshold, 255, 0).astype(np.uint8)
    return binary, best_threshold

binary_otsu, otsu_value = otsu_threshold(gray)

print("Otsu threshold value:", otsu_value)
display_image(binary_otsu, "Otsu threshold")

## 5. Adaptive thresholding

Adaptive thresholding uses local neighborhoods.

It is useful when lighting is uneven across the page.

In [ ]:
def add_lighting_gradient(image_rgb):
    if image_rgb is None:
        return None

    height, width = image_rgb.shape[:2]
    gradient = np.linspace(0.65, 1.15, width)
    gradient = np.tile(gradient, (height, 1))
    out = image_rgb.astype(np.float32)

    for channel in range(3):
        out[:, :, channel] *= gradient

    return np.clip(out, 0, 255).astype(np.uint8)

gradient_rgb = add_lighting_gradient(clean_rgb)
gradient_gray = to_grayscale(gradient_rgb)

display_image(gradient_rgb, "Uneven lighting image")

In [ ]:
def adaptive_threshold(gray_image, block_size=35, c_value=11):
    if gray_image is None:
        return None

    if block_size % 2 == 0:
        block_size += 1

    if CV2_AVAILABLE:
        return cv2.adaptiveThreshold(
            gray_image,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            block_size,
            c_value
        )

    # Simple fallback: local mean threshold.
    pad = block_size // 2
    padded = np.pad(gray_image, pad, mode="edge")
    output = np.zeros_like(gray_image)

    for y in range(gray_image.shape[0]):
        for x in range(gray_image.shape[1]):
            window = padded[y:y + block_size, x:x + block_size]
            local_threshold = np.mean(window) - c_value
            output[y, x] = 255 if gray_image[y, x] > local_threshold else 0

    return output.astype(np.uint8)

global_on_gradient = global_threshold(gradient_gray, threshold_value=180)
adaptive_on_gradient = adaptive_threshold(gradient_gray, block_size=35, c_value=11)

print("Global threshold on uneven lighting:")
display_image(global_on_gradient)

print("Adaptive threshold on uneven lighting:")
display_image(adaptive_on_gradient)

## 6. Denoising

Noise can create false characters and broken letters.

Denoising smooths the image while trying to keep text edges.

In [ ]:
def add_noise(image_rgb, noise_std=28, seed=42):
    if image_rgb is None:
        return None

    rng = np.random.default_rng(seed)
    noise = rng.normal(0, noise_std, image_rgb.shape)
    noisy = image_rgb.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

noisy_rgb = add_noise(clean_rgb, noise_std=32)
noisy_gray = to_grayscale(noisy_rgb)

display_image(noisy_rgb, "Noisy image")

In [ ]:
def denoise_image(gray_image):
    if gray_image is None:
        return None

    if CV2_AVAILABLE:
        return cv2.fastNlMeansDenoising(gray_image, None, h=18, templateWindowSize=7, searchWindowSize=21)

    if PIL_AVAILABLE:
        pil = Image.fromarray(gray_image.astype(np.uint8))
        return np.array(pil.filter(ImageFilter.MedianFilter(size=3)))

    return gray_image

denoised_gray = denoise_image(noisy_gray)

print("Noisy stats:", image_stats(noisy_gray))
print("Denoised stats:", image_stats(denoised_gray))

display_image(denoised_gray, "Denoised grayscale")

In [ ]:
noisy_otsu, noisy_otsu_value = otsu_threshold(noisy_gray)
denoised_otsu, denoised_otsu_value = otsu_threshold(denoised_gray)

print("Noisy Otsu value:", noisy_otsu_value)
display_image(noisy_otsu)

print("Denoised Otsu value:", denoised_otsu_value)
display_image(denoised_otsu)

## 7. Morphological operations

Morphology edits shapes in a binary image.

Opening can remove tiny noise. Closing can connect small gaps in letters.

In [ ]:
def morphology_open(binary_image, kernel_size=2):
    if binary_image is None:
        return None

    if CV2_AVAILABLE:
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        return cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, kernel)

    return binary_image

def morphology_close(binary_image, kernel_size=2):
    if binary_image is None:
        return None

    if CV2_AVAILABLE:
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        return cv2.morphologyEx(binary_image, cv2.MORPH_CLOSE, kernel)

    return binary_image

opened = morphology_open(denoised_otsu, kernel_size=2)
closed = morphology_close(denoised_otsu, kernel_size=2)

print("Opened image:")
display_image(opened)

print("Closed image:")
display_image(closed)

## 8. Deskewing

Skew means the document is tilted.

Deskewing rotates the image back so text lines become horizontal.

In [ ]:
def rotate_image_rgb(image_rgb, angle_degrees):
    if image_rgb is None:
        return None

    if CV2_AVAILABLE:
        height, width = image_rgb.shape[:2]
        center = (width // 2, height // 2)
        matrix = cv2.getRotationMatrix2D(center, angle_degrees, 1.0)
        rotated = cv2.warpAffine(image_rgb, matrix, (width, height), borderValue=(255, 255, 255))
        return rotated

    if PIL_AVAILABLE:
        pil = rgb_array_to_pil(image_rgb)
        return np.array(pil.rotate(angle_degrees, expand=False, fillcolor="white"))

    return image_rgb

skewed_rgb = rotate_image_rgb(clean_rgb, angle_degrees=7)
skewed_gray = to_grayscale(skewed_rgb)

display_image(skewed_rgb, "Skewed image")

In [ ]:
def estimate_skew_angle(gray_image):
    if gray_image is None:
        return 0.0

    if CV2_AVAILABLE:
        _, binary = cv2.threshold(gray_image, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        coords = np.column_stack(np.where(binary > 0))

        if len(coords) < 10:
            return 0.0

        angle = cv2.minAreaRect(coords)[-1]

        if angle < -45:
            angle = 90 + angle

        return float(angle)

    # Fallback: no reliable skew estimate without OpenCV.
    return 0.0

estimated_angle = estimate_skew_angle(skewed_gray)
print("Estimated skew angle:", round(estimated_angle, 2))

In [ ]:
def deskew_image_rgb(image_rgb):
    if image_rgb is None:
        return None, 0.0

    gray_image = to_grayscale(image_rgb)
    angle = estimate_skew_angle(gray_image)

    # Rotate in the opposite direction.
    corrected = rotate_image_rgb(image_rgb, angle_degrees=angle)
    return corrected, angle

deskewed_rgb, correction_angle = deskew_image_rgb(skewed_rgb)

print("Correction angle:", round(correction_angle, 2))
display_image(deskewed_rgb, "Deskewed image")

## 9. Build a preprocessing pipeline

A simple OCR preprocessing pipeline often follows this order.

Grayscale, denoise, threshold, and optionally deskew.

In [ ]:
def preprocess_for_ocr(image_rgb, use_denoise=True, threshold_method="otsu", deskew=False):
    if image_rgb is None:
        return None

    working = image_rgb

    if deskew:
        working, angle = deskew_image_rgb(working)
    else:
        angle = 0.0

    gray_image = to_grayscale(working)

    if use_denoise:
        gray_image = denoise_image(gray_image)

    if threshold_method == "otsu":
        binary, threshold_value = otsu_threshold(gray_image)
    elif threshold_method == "adaptive":
        binary = adaptive_threshold(gray_image)
        threshold_value = "adaptive"
    elif threshold_method == "global":
        binary = global_threshold(gray_image, threshold_value=180)
        threshold_value = 180
    else:
        binary = gray_image
        threshold_value = None

    return {
        "image": binary,
        "deskew_angle": angle,
        "threshold_method": threshold_method,
        "threshold_value": threshold_value
    }

pipeline_result = preprocess_for_ocr(noisy_rgb, use_denoise=True, threshold_method="otsu", deskew=False)

pprint({k: v for k, v in pipeline_result.items() if k != "image"})
display_image(pipeline_result["image"], "Pipeline output")

In [ ]:
pipeline_variants = []

for method in ["global", "otsu", "adaptive"]:
    result = preprocess_for_ocr(gradient_rgb, use_denoise=True, threshold_method=method, deskew=False)
    pipeline_variants.append({
        "method": method,
        "threshold_value": result["threshold_value"],
        "mean_pixel": round(float(np.mean(result["image"])), 2)
    })

pd.DataFrame(pipeline_variants)

## 10. OCR quality proxy

Without running a real OCR engine, we can still compare image quality roughly.

This proxy checks contrast and how much of the image is dark text.

In [ ]:
def quality_proxy(binary_or_gray_image):
    if binary_or_gray_image is None:
        return {}

    dark_ratio = float(np.mean(binary_or_gray_image < 128))
    contrast = float(np.std(binary_or_gray_image))

    if 0.01 <= dark_ratio <= 0.35 and contrast > 40:
        rating = "likely good"
    elif dark_ratio > 0.45:
        rating = "too dark or noisy"
    elif dark_ratio < 0.005:
        rating = "too light or missing text"
    else:
        rating = "needs inspection"

    return {
        "dark_ratio": round(dark_ratio, 4),
        "contrast": round(contrast, 2),
        "rating": rating
    }

quality_results = {
    "gray": quality_proxy(gray),
    "global": quality_proxy(binary_global),
    "otsu": quality_proxy(binary_otsu),
    "adaptive_gradient": quality_proxy(adaptive_on_gradient),
    "denoised_otsu": quality_proxy(denoised_otsu),
}

pprint(quality_results)

## Tricky bits

Preprocessing is not always better.

Too much denoising can blur letters. Too much thresholding can erase light text. Always compare outputs on real OCR results when possible.

In [ ]:
preprocessing_mistakes = pd.DataFrame([
    {
        "mistake": "Using one threshold for every document",
        "why_it_hurts": "Lighting and scan quality change",
        "better_option": "Test Otsu and adaptive thresholding"
    },
    {
        "mistake": "Over-denoising",
        "why_it_hurts": "Letters can become blurry",
        "better_option": "Use light denoising and compare OCR output"
    },
    {
        "mistake": "Ignoring skew",
        "why_it_hurts": "Tilted lines reduce OCR accuracy",
        "better_option": "Estimate angle and deskew"
    },
    {
        "mistake": "Judging only by visual beauty",
        "why_it_hurts": "Pretty images are not always best for OCR",
        "better_option": "Measure field-level extraction accuracy"
    },
])

preprocessing_mistakes

In [ ]:
def choose_preprocessing_problem(symptom):
    symptom = symptom.lower()
    if "uneven" in symptom or "shadow" in symptom:
        return "Try adaptive thresholding."
    if "noise" in symptom or "speckles" in symptom:
        return "Try denoising and opening."
    if "tilted" in symptom or "skew" in symptom:
        return "Try deskewing."
    if "light" in symptom or "faint" in symptom:
        return "Adjust thresholding or improve contrast."
    return "Start with grayscale plus Otsu thresholding."

symptoms = [
    "The image has shadows and uneven lighting",
    "There are many tiny speckles",
    "The receipt is tilted",
    "The text is very light"
]

for symptom in symptoms:
    print(symptom, "=>", choose_preprocessing_problem(symptom))

## Trick questions

1. Does preprocessing always improve OCR?

<details>
<summary>Answer</summary>

No. Bad preprocessing can remove letters, blur text, or create noise. Always compare results.

</details>

2. When is adaptive thresholding useful?

<details>
<summary>Answer</summary>

When the image has uneven lighting, shadows, or different brightness across the page.

</details>

3. What does Otsu thresholding do?

<details>
<summary>Answer</summary>

It automatically chooses a threshold value based on the pixel intensity distribution.

</details>

4. Why deskew before OCR?

<details>
<summary>Answer</summary>

OCR engines usually read horizontal text better than tilted text.

</details>

5. Why convert to grayscale first?

<details>
<summary>Answer</summary>

It simplifies the image and keeps brightness information that is useful for text detection.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Convert the clean RGB image to grayscale.

exercise_gray = ___

assert exercise_gray is not None
assert len(exercise_gray.shape) == 2
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Apply global thresholding with threshold value 180.

exercise_binary = ___

assert exercise_binary is not None
assert set(np.unique(exercise_binary)).issubset({0, 255})
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Apply Otsu thresholding and capture both outputs.

exercise_otsu, exercise_otsu_value = ___

assert exercise_otsu is not None
assert isinstance(exercise_otsu_value, (int, float, np.integer, np.floating))
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Apply adaptive thresholding to the gradient image.

exercise_adaptive = ___

assert exercise_adaptive is not None
assert set(np.unique(exercise_adaptive)).issubset({0, 255})
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Denoise the noisy grayscale image.

exercise_denoised = ___

assert exercise_denoised is not None
assert exercise_denoised.shape == noisy_gray.shape
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Estimate the skew angle of the skewed grayscale image.

angle = ___

assert isinstance(angle, float)
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Run the full preprocessing pipeline on the noisy image.

result = ___

assert isinstance(result, dict)
assert "image" in result
assert "threshold_method" in result
print("Exercise 7 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
exercise_gray = to_grayscale(clean_rgb)
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
exercise_binary = global_threshold(exercise_gray, threshold_value=180)
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
exercise_otsu, exercise_otsu_value = otsu_threshold(exercise_gray)
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
exercise_adaptive = adaptive_threshold(gradient_gray, block_size=35, c_value=11)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
exercise_denoised = denoise_image(noisy_gray)
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
angle = estimate_skew_angle(skewed_gray)
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
result = preprocess_for_ocr(noisy_rgb, use_denoise=True, threshold_method="otsu", deskew=False)
```

</details>

## Cumulative review exercises

These mix topics from Days 13 to 22. Fill in `___` and run each cell.

In [ ]:
# Review 1: Hugging Face
# Fill the quick helper for pretrained tasks.

hf_helper = ___

assert hf_helper == "pipeline"
print("Review 1 passed.")

In [ ]:
# Review 2: Fine-tuning BERT
# Pick the common Hugging Face training class.

training_class = ___

assert training_class == "Trainer"
print("Review 2 passed.")

In [ ]:
# Review 3: Complaint classification
# Create a label mapping.

label_to_id = ___

assert isinstance(label_to_id, dict)
assert len(label_to_id) >= 3
assert all(isinstance(value, int) for value in label_to_id.values())
print("Review 3 passed.")

In [ ]:
# Review 4: OpenAI API
# Fill the standard chat roles.

roles = ___

assert roles == ["system", "user", "assistant"]
print("Review 4 passed.")

In [ ]:
# Review 5: Ollama
# Fill the default local generate endpoint.

ollama_generate_url = ___

assert ollama_generate_url == "http://localhost:11434/api/generate"
print("Review 5 passed.")

In [ ]:
# Review 6: Prompt engineering
# Choose the prompting style that includes examples.

prompt_style = ___

assert prompt_style.lower() == "few-shot"
print("Review 6 passed.")

In [ ]:
# Review 7: Structured output
# Parse JSON text.

json_text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___

assert parsed["clicks"] == 100
print("Review 7 passed.")

In [ ]:
# Review 8: Information extraction
# Calculate conversion rate.

record = {"clicks": 1000, "conversions": 75}
conversion_rate = ___

assert abs(conversion_rate - 0.075) < 1e-9
print("Review 8 passed.")

In [ ]:
# Review 9: Tesseract basics
# Choose the Tesseract language code for Turkish.

turkish_lang_code = ___

assert turkish_lang_code == "tur"
print("Review 9 passed.")

In [ ]:
# Review 10: EasyOCR
# Create an EasyOCR language list for English and German.

easyocr_languages = ___

assert easyocr_languages == ["en", "de"] or easyocr_languages == ["de", "en"]
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
hf_helper = "pipeline"

# Review 2
training_class = "Trainer"

# Review 3
label_to_id = {"billing": 0, "delivery": 1, "technical": 2}

# Review 4
roles = ["system", "user", "assistant"]

# Review 5
ollama_generate_url = "http://localhost:11434/api/generate"

# Review 6
prompt_style = "few-shot"

# Review 7
parsed = json.loads(json_text)

# Review 8
conversion_rate = record["conversions"] / record["clicks"]

# Review 9
turkish_lang_code = "tur"

# Review 10
easyocr_languages = ["en", "de"]
```

</details>

In [ ]:
cheat_sheet = '''
DAY 23 CHEAT SHEET: OPENCV PREPROCESSING

Why preprocessing:
- OCR works better when text is clear, horizontal, and high contrast.

Grayscale:
- Converts RGB image to one brightness channel.
- Useful before thresholding.

Thresholding:
- Global threshold: one fixed cutoff.
- Otsu threshold: automatic cutoff.
- Adaptive threshold: local cutoff for uneven lighting.

Denoising:
- Removes speckles and noise.
- Too much denoising can blur letters.

Morphology:
- Opening can remove tiny noise.
- Closing can connect small gaps.

Deskewing:
- Estimate text angle.
- Rotate image back to horizontal.

Common pipeline:
1. RGB image
2. Deskew if needed
3. Grayscale
4. Denoise
5. Threshold
6. OCR
7. Field extraction and validation
'''

print(cheat_sheet)

## Next up: Day 24 — OcrLlmPipeline

You will connect image preprocessing, OCR, LLM extraction, and structured JSON handling.